# Few-Shot Experiment — Notebook 10
## VLM Medical VQA Benchmark

**Research Question:** Does in-context learning (few-shot prompting) close the performance gap
between generalist and medical VLMs without any fine-tuning?

| Run | Set `RUN_CONFIG` to | Model | Notes |
|---|---|---|---|
| A | `gemma3_0shot` | Gemma-3-4B-IT | Baseline, ~30 min |
| B | `gemma3_1shot` | Gemma-3-4B-IT | 1 example, ~35 min |
| C | `gemma3_3shot` | Gemma-3-4B-IT | 3 examples, ~45 min |
| D | `llava_0shot` | LLaVA-1.6-Mistral-7B | Baseline, 4-bit NF4, ~45 min |
| E | `llava_1shot` | LLaVA-1.6-Mistral-7B | 1 example, 4-bit NF4, ~55 min |
| F | `llava_3shot` | LLaVA-1.6-Mistral-7B | 3 examples, 4-bit NF4, ~70 min |

**Instructions:**
1. Set `RUN_CONFIG` in Cell 1.
2. Add your HF token to Kaggle Secrets as `HF_TOKEN`.
3. Run all cells top-to-bottom. SLAKE images are downloaded automatically in Cell 3.
4. Download the output `.jsonl` from the Output tab and place in `outputs/_archive/fewshot_experiment/`.

## Cell 1 — Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  SET THIS BEFORE RUNNING
# ══════════════════════════════════════════════════════════════════
RUN_CONFIG = "llava_3shot"   # change to your desired run
# ══════════════════════════════════════════════════════════════════

import os

# SLAKE images will be downloaded and extracted here
SLAKE_IMGS_DIR = '/kaggle/working/slake_imgs'

CONFIGS = {
    "gemma3_0shot": {
        "model_id": "google/gemma-3-4b-it",
        "model_short": "Gemma-3-4B",
        "n_shot": 0,
        "use_4bit": False,
        "max_new_tokens": 64,
        "file_tag": "gemma3_4b__slake_0shot",
    },
    "gemma3_1shot": {
        "model_id": "google/gemma-3-4b-it",
        "model_short": "Gemma-3-4B",
        "n_shot": 1,
        "use_4bit": False,
        "max_new_tokens": 64,
        "file_tag": "gemma3_4b__slake_1shot",
    },
    "gemma3_3shot": {
        "model_id": "google/gemma-3-4b-it",
        "model_short": "Gemma-3-4B",
        "n_shot": 3,
        "use_4bit": False,
        "max_new_tokens": 64,
        "file_tag": "gemma3_4b__slake_3shot",
    },
    "llava_0shot": {
        "model_id": "llava-hf/llava-v1.6-mistral-7b-hf",
        "model_short": "LLaVA-1.6-7B",
        "n_shot": 0,
        "use_4bit": True,
        "max_new_tokens": 64,
        "file_tag": "llava16_7b__slake_0shot",
    },
    "llava_1shot": {
        "model_id": "llava-hf/llava-v1.6-mistral-7b-hf",
        "model_short": "LLaVA-1.6-7B",
        "n_shot": 1,
        "use_4bit": True,
        "max_new_tokens": 64,
        "file_tag": "llava16_7b__slake_1shot",
    },
    "llava_3shot": {
        "model_id": "llava-hf/llava-v1.6-mistral-7b-hf",
        "model_short": "LLaVA-1.6-7B",
        "n_shot": 3,
        "use_4bit": True,
        "max_new_tokens": 64,
        "file_tag": "llava16_7b__slake_3shot",
    },
}

cfg            = CONFIGS[RUN_CONFIG]
MODEL_ID       = cfg["model_id"]
MODEL_SHORT    = cfg["model_short"]
N_SHOT         = cfg["n_shot"]
USE_4BIT       = cfg["use_4bit"]
MAX_NEW_TOKENS = cfg["max_new_tokens"]
FILE_TAG       = cfg["file_tag"]
OUTPUT_DIR     = "/kaggle/working"
OUT_PATH       = f"{OUTPUT_DIR}/{FILE_TAG}.jsonl"

print(f"Run config : {RUN_CONFIG}")
print(f"Model      : {MODEL_ID}")
print(f"N-shot     : {N_SHOT}")
print(f"4-bit NF4  : {USE_4BIT}")
print(f"Output     : {OUT_PATH}")

## Cell 2 — GPU Check + Install

In [ ]:
!nvidia-smi
!pip install -q transformers==4.51.3 accelerate bitsandbytes datasets sacrebleu tqdm

## Cell 3 — Download SLAKE Images
Downloads `imgs.zip` directly from HuggingFace and extracts to `/kaggle/working/slake_imgs/`.
Skip this cell on re-runs if images are already extracted.

In [ ]:
import zipfile, subprocess

os.makedirs(SLAKE_IMGS_DIR, exist_ok=True)

zip_path = f'{SLAKE_IMGS_DIR}/imgs.zip'
url = 'https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip'

# Only download if zip not already there (saves time on re-runs)
if not os.path.exists(zip_path):
    print('Downloading SLAKE images ...')
    subprocess.run(['curl', '-L', url, '-o', zip_path], capture_output=True)
    print('Download complete.')
else:
    print('Zip already exists, skipping download.')

# Extract
print('Extracting ...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(SLAKE_IMGS_DIR)

# Count images and detect the actual image root
imgs = []
for root, dirs, files in os.walk(SLAKE_IMGS_DIR):
    for f in files:
        if f.endswith(('.jpg', '.png', '.jpeg')):
            imgs.append(os.path.join(root, f))
print(f'Extracted {len(imgs)} images.')

# The zip extracts to slake_imgs/imgs/<subfolder>/source.jpg
# SLAKE img_name field is like 'xmlab102/source.jpg'
# So the base to prepend is SLAKE_IMGS_DIR/imgs/
SLAKE_IMG_BASE = os.path.join(SLAKE_IMGS_DIR, 'imgs')
if not os.path.isdir(SLAKE_IMG_BASE):
    # Fallback: imgs extracted directly into SLAKE_IMGS_DIR
    SLAKE_IMG_BASE = SLAKE_IMGS_DIR

print(f'Image base dir : {SLAKE_IMG_BASE}')
print(f'Exists         : {os.path.isdir(SLAKE_IMG_BASE)}')

# Quick sanity check — show first 3 image paths
for p in imgs[:3]:
    print(f'  {p}')

## Cell 4 — Imports & Device

In [ ]:
import json, re, random
import torch
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')
if device == 'cuda':
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  {props.total_memory // 1024**2} MB')

## Cell 5 — Hugging Face Authentication

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('HF login successful.')
except Exception as e:
    print(f'HF login failed: {e}')

## Cell 6 — Load SLAKE & Build Stratified 200-Sample Subset

**SLAKE field notes:**
- `content_type`: `'Modality'`, `'Organ'`, `'Abnormality'` (full words, title case — NOT abbreviations)
- `answer_type`: `'OPEN'` or `'CLOSED'`
- `img_name`: relative path like `xmlab102/source.jpg` — resolved against `SLAKE_IMG_BASE`

The 200-sample subset is stratified across 6 buckets (3 content types × 2 answer types, ~33 per bucket).
**All 6 runs use the exact same 200 samples** (fixed seed=42).

In [ ]:
from collections import Counter

# ── Load HF metadata ──────────────────────────────────────────────────────────
ds_raw = load_dataset('BoKelvin/SLAKE')

test_all  = [s for s in ds_raw['test']  if s.get('q_lang') == 'en']
train_all = [s for s in ds_raw['train'] if s.get('q_lang') == 'en']

print(f'SLAKE EN test  : {len(test_all)}')
print(f'SLAKE EN train : {len(train_all)}')

# ── Field accessors ───────────────────────────────────────────────────────────
def get_content_type(s):
    """Returns 'Modality', 'Organ', 'Abnormality', etc. (exact SLAKE values)."""
    return str(s.get('content_type', '') or '').strip()

def is_closed(s):
    return str(s.get('answer_type', '')).upper() == 'CLOSED'

# ── Image loader ──────────────────────────────────────────────────────────────
def load_image(sample):
    img_name = sample.get('img_name', '')   # e.g. 'xmlab102/source.jpg'
    img_path = os.path.join(SLAKE_IMG_BASE, img_name)
    return Image.open(img_path).convert('RGB')

def image_exists(sample):
    img_name = sample.get('img_name', '')
    return os.path.exists(os.path.join(SLAKE_IMG_BASE, img_name))

# Verify image loading
print('\nVerifying image loading ...')
test_img = load_image(test_all[0])
print(f'OK — size={test_img.size}, mode={test_img.mode}')

# Distribution check
ct_counts = Counter(get_content_type(s) for s in test_all)
print('\nContent type distribution (test EN):')
for ct, n in sorted(ct_counts.items(), key=lambda x: -x[1]):
    print(f'  {ct:15s}: {n}')

# ── Stratified 200-sample subset ──────────────────────────────────────────────
TARGET_TYPES       = ['Modality', 'Organ', 'Abnormality']
SAMPLES_PER_BUCKET = 33
SEED               = 42

rng = random.Random(SEED)

buckets = {}
for ctype in TARGET_TYPES:
    for ans_type in ['CLOSED', 'OPEN']:
        pool = [
            s for s in test_all
            if get_content_type(s) == ctype
            and ('CLOSED' if is_closed(s) else 'OPEN') == ans_type
        ]
        rng.shuffle(pool)
        buckets[(ctype, ans_type)] = pool[:SAMPLES_PER_BUCKET]

test_subset = [s for pool in buckets.values() for s in pool]

# Top up to 200 if any bucket was undersized
already = {id(s) for s in test_subset}
remainder = [s for s in test_all
             if get_content_type(s) in TARGET_TYPES and id(s) not in already]
rng.shuffle(remainder)
test_subset.extend(remainder[:max(0, 200 - len(test_subset))])
test_subset = test_subset[:200]

print(f'\nFinal test subset : {len(test_subset)} questions')
print('Bucket breakdown:')
for (ct, at), pool in buckets.items():
    n_in = sum(
        1 for s in test_subset
        if get_content_type(s) == ct
        and ('CLOSED' if is_closed(s) else 'OPEN') == at
    )
    print(f'  {ct:12s} × {at:6s}: {n_in}')

closed_n = sum(1 for s in test_subset if is_closed(s))
print(f'Total closed: {closed_n}  |  Total open: {len(test_subset) - closed_n}')

## Cell 7 — Curate Few-Shot Exemplars From Training Split

One exemplar per content type (Modality → Organ → Abnormality), drawn from **training split only**.
Criteria: answer is 1–3 words, image file exists on disk.

In [ ]:
def is_clean_exemplar(sample):
    answer = str(sample.get('answer', '')).strip()
    return 0 < len(answer.split()) <= 3

rng_ex = random.Random(SEED + 1)
exemplars = []

for target_type in TARGET_TYPES:
    pool = [
        s for s in train_all
        if get_content_type(s) == target_type
        and is_clean_exemplar(s)
        and image_exists(s)
    ]
    rng_ex.shuffle(pool)

    if not pool:
        print(f'WARNING: No exemplar found for content_type="{target_type}"')
        continue

    chosen = pool[0]
    exemplars.append({
        'content_type': target_type,
        'question':     chosen['question'],
        'answer':       str(chosen['answer']).strip(),
        'image':        load_image(chosen),
        'is_closed':    is_closed(chosen),
    })
    print(f'  [{target_type:12s}] Q="{chosen["question"]}"  A="{chosen["answer"]}"')

print(f'\nTotal exemplars prepared : {len(exemplars)}')
print(f'1-shot uses              : exemplars[0] ({exemplars[0]["content_type"] if exemplars else "N/A"})')
print(f'3-shot uses              : all 3 ({[e["content_type"] for e in exemplars]})')

if N_SHOT > 0 and len(exemplars) < N_SHOT:
    raise RuntimeError(f'Need {N_SHOT} exemplars but only have {len(exemplars)}.')

## Cell 8 — Prompt Builders

In [ ]:
def build_prompt(question: str, is_closed_q: bool) -> str:
    """Exact prompt from vqa-evaluation-4b.ipynb (MedGemma paper protocol)."""
    prefix = 'Answer the question with yes or no. ' if is_closed_q else ''
    return (
        f"{prefix}{question} "
        f"You may write out your argument before stating your final very short, "
        f"definitive, and concise answer (if possible, a single word) "
        f"X in the format 'Final Answer: X'"
    )

def extract_answer(text: str) -> str:
    import string
    clean = re.sub(r'\*+', '', text).strip()
    match = re.search(r'[Ff]inal\s+[Aa]nswer\s*:\s*(.+)', clean, re.DOTALL)
    if match:
        ans = match.group(1).strip().split('\n')[0].strip()
        return ans.strip(string.punctuation + ' ')
    lines = [l.strip() for l in clean.split('\n') if l.strip()]
    return lines[-1] if lines else clean

print('Prompt builders OK.')
s0 = test_subset[0]
print(build_prompt(s0['question'], is_closed(s0)))


## Cell 9 — Load Model & Processor

In [ ]:
dtype = torch.bfloat16 if device == 'cuda' else torch.float32

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )
    print('Loading in 4-bit NF4 ...')
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True,
    )
else:
    print(f'Loading in {dtype} ...')
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, torch_dtype=dtype,
        device_map='auto', trust_remote_code=True,
    )

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()
print(f'Model loaded : {MODEL_ID}')
param_b = sum(p.numel() for p in model.parameters()) / 1e9
print(f'Parameters   : {param_b:.2f}B')

## Cell 10 — Inference Loop

- **0-shot:** Standard single-image inference (matches existing benchmark)
- **1-shot:** Modality exemplar prepended as a prior conversation turn
- **3-shot:** Modality → Organ → Abnormality exemplars prepended

Each exemplar turn: `[user: image + question]` → `[assistant: correct answer]`

In [ ]:
# ── Resume support — only valid records count as done ─────────────────────────
completed = {}
if os.path.exists(OUT_PATH):
    with open(OUT_PATH) as f:
        for line in f:
            try:
                r = json.loads(line)
                is_valid = bool(r.get('error')) or bool(r.get('raw_output', '').strip())
                if is_valid:
                    completed[r['idx']] = r
            except:
                pass
    total_lines = sum(1 for _ in open(OUT_PATH))
    bad = total_lines - len(completed)
    print(f'Resuming: {len(completed)} valid / {total_lines} total ({bad} empty records will be re-run).')
    if bad > 0:
        valid_records = list(completed.values())
        with open(OUT_PATH, 'w') as fw:
            for vr in sorted(valid_records, key=lambda x: x['idx']):
                fw.write(json.dumps(vr) + '\n')
        print(f'  Rewrote file: kept {len(valid_records)}, removed {bad} empty records.')

active_exemplars = exemplars[:N_SHOT]
errors = 0
f_out  = open(OUT_PATH, 'a')

for i, sample in enumerate(tqdm(test_subset, desc=RUN_CONFIG)):
    if i in completed:
        continue

    try:
        target_image    = load_image(sample)
        target_question = sample['question']
        target_answer   = str(sample['answer']).strip()
        target_closed   = is_closed(sample)
        target_ctype    = get_content_type(sample)

        question_text = build_prompt(target_question, target_closed)

        # ── Build messages — place image tag without the raw image data ──
        messages   = []
        all_images = []   

        for ex in active_exemplars:
            ex_prompt = build_prompt(ex['question'], ex['is_closed'])
            messages.append({
                'role': 'user',
                'content': [
                    {'type': 'image'},   # ← image tag only
                    {'type': 'text',  'text':  ex_prompt},
                ]
            })
            messages.append({
                'role': 'assistant',
                'content': [
                    {'type': 'text', 'text': ex['answer']} # ← Updated to dict format
                ],
            })
            all_images.append(ex['image'])

        messages.append({
            'role': 'user',
            'content': [
                {'type': 'image'},   # ← image tag only
                {'type': 'text',  'text':  question_text},
            ]
        })
        all_images.append(target_image)

        # ── Tokenize ───────────────────────────────────────────────────────────
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        if len(all_images) == 1:
            inputs = processor(
                text=text, images=all_images[0], return_tensors='pt'
            ).to(device)
        else:
            inputs = processor(
                text=text, images=all_images, return_tensors='pt'
            ).to(device)

        # ── Generate ───────────────────────────────────────────────────────────
        with torch.inference_mode():
            output_ids = model.generate(**inputs, max_new_tokens=300, do_sample=False, temperature=1.0)

        input_len  = inputs['input_ids'].shape[-1]
        raw        = processor.decode(
            output_ids[0][input_len:], skip_special_tokens=True
        ).strip()
        prediction = extract_answer(raw)

        record = {
            'idx':          i,
            'question':     target_question,
            'ground_truth': target_answer,
            'prediction':   prediction,
            'raw_output':   raw,
            'is_closed':    target_closed,
            'content_type': target_ctype,
            'n_shot':       N_SHOT,
            'model':        MODEL_ID,
            'model_short':  MODEL_SHORT,
        }

    except Exception as e:
        errors += 1
        print(f'  Error at idx={i}: {e}')
        record = {
            'idx': i, 'question': '', 'ground_truth': '',
            'prediction': '', 'raw_output': '',
            'is_closed': False, 'content_type': '',
            'n_shot': N_SHOT, 'model': MODEL_ID,
            'model_short': MODEL_SHORT, 'error': str(e),
        }

    f_out.write(json.dumps(record) + '\n')
    f_out.flush()

f_out.close()
print(f'\nDone. {len(test_subset)} questions  |  {errors} errors  ->  {OUT_PATH}')

## Cell 11 — Quick Metrics Preview

In [ ]:
def norm(t):
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', ' ', str(t).lower())).strip()

def token_f1(pred, gt):
    from collections import Counter
    p, g = norm(pred).split(), norm(gt).split()
    if not p or not g: return 0.0
    pc, gc = Counter(p), Counter(g)
    common = sum((pc & gc).values())
    if not common: return 0.0
    pr = common / len(p); rc = common / len(g)
    return 2 * pr * rc / (pr + rc)

def closed_acc(recs):
    if not recs: return 0.0
    correct = sum(
        1 for r in recs if
        norm(r['prediction']) == norm(r['ground_truth']) or
        ('yes' in norm(r['prediction']) and 'yes' in norm(r['ground_truth'])) or
        ('no'  in norm(r['prediction']) and 'no'  in norm(r['ground_truth']))
    )
    return correct / len(recs)

records  = [json.loads(l) for l in open(OUT_PATH) if 'error' not in l]
closed_r = [r for r in records if r.get('is_closed')]
open_r   = [r for r in records if not r.get('is_closed')]
f1_all   = [token_f1(r['prediction'], r['ground_truth']) for r in records]
f1_open  = [token_f1(r['prediction'], r['ground_truth']) for r in open_r]

print(f'Run        : {RUN_CONFIG}  ({N_SHOT}-shot)')
print(f'Model      : {MODEL_SHORT}')
print(f'Records    : {len(records)}')
print(f'Overall F1 : {sum(f1_all)/len(f1_all)*100:.2f}%')
if closed_r:
    print(f'Closed Acc : {closed_acc(closed_r)*100:.2f}%  (N={len(closed_r)})')
if open_r:
    print(f'Open F1    : {sum(f1_open)/len(f1_open)*100:.2f}%  (N={len(open_r)})')

print('\nSample predictions:')
for r in records[:5]:
    ct = r.get('content_type', '?')
    print(f'  [{ct:11s}] GT: {r["ground_truth"]:<15}  Pred: {r["prediction"][:45]}')

print(f'\nFile: {OUT_PATH}')

## Cell 12 — Download & Next Steps

1. Go to the **Output** tab → download the `.jsonl` file.
2. Place it in `outputs/_archive/fewshot_experiment/` with **this exact filename:**

| Run | Config | Filename |
|---|---|---|
| A | `gemma3_0shot` | `gemma3_4b__slake_0shot.jsonl` |
| B | `gemma3_1shot` | `gemma3_4b__slake_1shot.jsonl` |
| C | `gemma3_3shot` | `gemma3_4b__slake_3shot.jsonl` |
| D | `llava_0shot` | `llava16_7b__slake_0shot.jsonl` |
| E | `llava_1shot` | `llava16_7b__slake_1shot.jsonl` |
| F | `llava_3shot` | `llava16_7b__slake_3shot.jsonl` |

3. Once all 6 files are ready, tell the agent — it will run `scripts/fewshot_analysis.py`.